# 07 — Comprehensive Evaluation & Scientific Ablation Studies
**Goal**: Execute the full AAAI V&V evaluation matrix and answer all research questions (RQ1 to RQ7):
- **RQ1**: Can SLMs do agentic debugging? (Turns $K=1, 3, 5$)
- **RQ2**: Does real execution feedback outperform random feedback?
- **RQ3**: Does RL (PPO/DPO) beat SFT and zero-shot baselines?
- **RQ4**: Do SLMs require dense rewards vs binary pass/fail rewards?
- **RQ5**: Is DPO competitive with PPO in performance and efficiency?
- **RQ6**: At what turn count $K$ does debugging performance plateau?
- **RQ7**: Does training generalize out-of-distribution to HumanEval and MBPP?

---

## Step 1: Setup & Benchmark Loading

In [ ]:
import os
import torch
import pandas as pd
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import agentic_debug_loop, agentic_loop_no_feedback
from src.evaluation.metrics import evaluate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Loading evaluation benchmarks (HumanEval & MBPP)...")
humaneval = load_dataset('openai_humaneval', split='test')
mbpp = load_dataset('mbpp', split='test')

print(f"Loaded HumanEval: {len(humaneval)} test cases | MBPP: {len(mbpp)} test cases.")

## Step 2: Main Evaluation Grid — Zero-Shot vs SFT vs PPO vs DPO (RQ3 & RQ7)
Computes Pass@1, Fix@3, and Fix@5 on HumanEval and MBPP across all trained models.

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
results_grid = []

checkpoints_to_eval = [
    ("Zero-Shot Base", MODEL_NAME),
    ("SFT Baseline", "./checkpoints/sft/final"),
    ("PPO Model", "./checkpoints/ppo/final"),
    ("DPO Model", "./checkpoints/dpo/final"),
]

for label, path in checkpoints_to_eval:
    actual_path = path if (path == MODEL_NAME or os.path.exists(path)) else MODEL_NAME
    print(f"\n================ Evaluatng: {label} ({actual_path}) ================")
    model, tokenizer = load_model_and_tokenizer(model_name=actual_path, load_in_4bit=True)
    
    for k in [1, 3, 5]:
        he_res = evaluate(model, tokenizer, humaneval.select(range(min(20, len(humaneval)))), K=k, label=f"{label} HE K={k}")
        mbpp_res = evaluate(model, tokenizer, mbpp.select(range(min(20, len(mbpp)))), K=k, label=f"{label} MBPP K={k}")
        
        results_grid.append({
            'Model': label,
            'K': k,
            'HumanEval Pass@1': f"{he_res['pass_at_1']:.2%}",
            f'HumanEval Fix@{k}': f"{he_res.get(f'fix_at_{k}', 0):.2%}",
            'MBPP Pass@1': f"{mbpp_res['pass_at_1']:.2%}",
            f'MBPP Fix@{k}': f"{mbpp_res.get(f'fix_at_{k}', 0):.2%}",
        })

df_results = pd.DataFrame(results_grid)
print("\n================ AAAI Evaluation Grid Results ================")
print(df_results.to_string(index=False))

## Step 3: Scientific Ablations (RQ2, RQ4, RQ6)
1. **RQ2 (Feedback Utility)**: Real execution traceback vs Random fake error injection (`agentic_loop_no_feedback`)
2. **RQ6 (Turn Count)**: Fix@K trajectory across $K \in \{1, 3, 5, 7\}$
3. **RQ4 (Reward Granularity)**: Dense partial-test reward vs Binary pass/fail reward

In [ ]:
print("\n--- Running RQ2 Feedback Ablation (Real vs Random Error) ---")
model, tokenizer = load_model_and_tokenizer(load_in_4bit=True)

res_real = evaluate(model, tokenizer, humaneval.select(range(10)), K=3, label="Real Feedback", debug_loop_fn=agentic_debug_loop)
res_fake = evaluate(model, tokenizer, humaneval.select(range(10)), K=3, label="Random Fake Feedback", debug_loop_fn=agentic_loop_no_feedback)

print(f"\nRQ2 Result: Real Feedback Fix@3={res_real.get('fix_at_3', 0):.2%} vs Random Fake Fix@3={res_fake.get('fix_at_3', 0):.2%}")

print("\n--- Running RQ6 K-Turn Plateau Ablation (K=1,3,5,7) ---")
k_trajectory = {}
for k_val in [1, 3, 5, 7]:
    res_k = evaluate(model, tokenizer, humaneval.select(range(10)), K=k_val, label=f"K={k_val}")
    k_trajectory[k_val] = res_k.get(f'fix_at_{k_val}', 0)

print("\nRQ6 Trajectory Results:")
for k_val, fix_rate in k_trajectory.items():
    print(f" - Fix@{k_val}: {fix_rate:.2%}")

print("\nAll evaluation and ablation studies finished successfully!")